# DRIVE Baseline — Attention Residual U-Net

**Target:** Vessel Dice ≥ 0.65 (DRIVE is the hardest of the 5 datasets — multi-blob vessel topology + only 20 labelled samples).

## Caveats
- Only the **training** split has vessel annotations publicly; DRIVE challenge withholds test labels.
- 20 labelled images → 15 / 3 / 2 patient-level split.
- Heavy augmentation in YAML compensates for low N. Early stop `patience=20`.
- `.gif` masks need PIL reader — handled via cfg `loader_reader: PILReader` and an in-notebook hotfix.

## Kaggle setup
1. Datasets: `andrewmvd/drive-digital-retinal-images-for-vessel-extraction` + `iteris-pkg`
2. Accelerator: **GPU T4 x2**, Persistence: **Files only**, Internet: **On**
3. Skip §3 dry-run for the real commit. Save Version → Save & Run All (Commit).

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Load config + locate DRIVE dataset

In [ ]:
from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config(str(PKG_ROOT / 'configs' / 'drive.yaml'))

# Locate DRIVE root — must contain training/1st_manual (vessel labels)
drive_candidates = [p for p in Path('/kaggle/input').rglob('DRIVE')
                    if p.is_dir() and (p / 'training' / '1st_manual').is_dir()]
if drive_candidates:
    cfg['data_root'] = str(drive_candidates[0].parent)
else:
    cfg['data_root'] = '/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Data root    : {cfg["data_root"]}')
print(f'Dataset      : {cfg["dataset"]}  ({cfg["modality"]})')
print(f'Image size   : {cfg["image_size"]}  in_channels={cfg["in_channels"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Normalise    : {cfg["normalize"]}   binarise labels: {cfg["binarize_labels"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}  patience {cfg["patience"]}')
print(f'Device       : {get_device()}')

## 2 · Loader hotfix — accept .gif masks

DRIVE vessel masks are `.gif`. MONAI's default ITKReader treats unknown extensions
as DICOM and crashes. We register a `PILReader` subclass that explicitly claims `.gif`.

By passing a pre-built **instance** as `cfg['loader_reader']` we also bypass the
`is_supported_format` import in older `iteris-pkg` versions that may still be on Kaggle.

In [ ]:
from monai.data import PILReader

class GIFPILReader(PILReader):
    @staticmethod
    def verify_suffix(filename):
        if isinstance(filename, (list, tuple)):
            return all(GIFPILReader.verify_suffix(f) for f in filename)
        return str(filename).lower().endswith(
            ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.gif', '.npy', '.npz')
        )

cfg['loader_reader'] = GIFPILReader()
print('✓ GIF reader registered.')

## 3 · OPTIONAL — Dry-run sanity check (~1 min)
Verifies the pipeline. **Skip this cell when committing the real run.**

In [ ]:
# === DRY RUN — comment out (or skip) for full training ===
cfg['epochs']              = 2
cfg['patience']            = 99
cfg['save_every_n_epochs'] = 0
cfg['batch_size']          = 2     # 15 train / 8 batch = 1 batch — drop to 2

from iteris.training import run_training
dry = run_training(cfg)
print(f'\n✓ Dry-run passed. Best val Dice (2 ep): {dry["best_dice"]:.4f}')

# Restore real hyperparameters for the full run
cfg = load_config(str(PKG_ROOT / 'configs' / 'drive.yaml'))
cfg['data_root']      = drive_candidates[0].parent.as_posix() if drive_candidates else cfg['data_root']
cfg['checkpoint_dir'] = '/kaggle/working'
cfg['loader_reader']  = GIFPILReader()
print('✓ cfg restored to YAML defaults.')

## 4 · Full training

In [ ]:
from iteris.training import run_training

result      = run_training(cfg)
model       = result['model']
history     = result['history']
best_dice   = result['best_dice']
test_loader = result['test_loader']
checkpoint  = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 5 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.65)

## 6 · Test-set evaluation
DRIVE test set is only 2 samples (10% of 20 labelled) — small but useful as a sanity check.

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 7 · Export predicted masks (init state for DRL agents)

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 8 · Qualitative grid (built-in, all test samples)

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=2)

## 9 · Stratified qualitative + Dice histogram
Per-sample Dice ranked. Adapted for DRIVE's small (n≈2) test set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
from iteris.utils  import get_device
from iteris.metrics import dice_score

device = get_device()
model.eval()
cmap = ListedColormap(cfg['class_colors'])

records = []
with torch.no_grad():
    for batch in test_loader:
        img  = batch['image'].to(device)
        gt   = batch['label'].to(device)
        pred = model(img).argmax(1)
        d_per_class = dice_score(pred, gt, cfg['num_classes']).cpu().numpy()
        records.append(dict(
            image = batch['image'][0].cpu().numpy(),
            gt    = batch['label'][0, 0].cpu().numpy(),
            pred  = pred[0].cpu().numpy(),
            dice  = float(d_per_class.mean()),
        ))

records.sort(key=lambda r: r['dice'])
n = len(records)

# Adaptive picks — DRIVE may have only 2 test samples
if n <= 2:
    picks = [(f'Sample {i+1}', records[i]) for i in range(n)]
elif n <= 4:
    picks = [('WORST', records[0]), ('BEST', records[-1])]
else:
    picks = [('WORST', records[0]),
             ('25%ile', records[n // 4]),
             ('MEDIAN', records[n // 2]),
             ('BEST',   records[-1])]

fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4 * len(picks)))
axes = np.atleast_2d(axes)
for row, (label, r) in enumerate(picks):
    img = r['image']
    is_rgb = img.ndim == 3 and img.shape[0] == 3
    img_show = img.transpose(1, 2, 0) if is_rgb else (img[0] if img.ndim == 3 else img)

    axes[row, 0].imshow(img_show, cmap=None if is_rgb else 'gray')
    axes[row, 0].set_title(f'{label}: Input'); axes[row, 0].axis('off')
    axes[row, 1].imshow(img_show, cmap=None if is_rgb else 'gray')
    axes[row, 1].imshow(r['gt'], cmap=cmap, alpha=0.5, vmin=0, vmax=cfg['num_classes']-1)
    axes[row, 1].set_title('Ground Truth'); axes[row, 1].axis('off')
    axes[row, 2].imshow(img_show, cmap=None if is_rgb else 'gray')
    axes[row, 2].imshow(r['pred'], cmap=cmap, alpha=0.5, vmin=0, vmax=cfg['num_classes']-1)
    axes[row, 2].set_title(f'Prediction  Dice={r["dice"]:.3f}'); axes[row, 2].axis('off')

plt.suptitle(f'{cfg["dataset"]} — stratified qualitative', fontsize=13)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["dataset"].lower()}_stratified.png', dpi=150)
plt.show()

if n >= 3:
    fig, ax = plt.subplots(figsize=(9, 4))
    dices = [r['dice'] for r in records]
    ax.hist(dices, bins=min(20, n), color='C0', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(dices),   color='red',   linestyle='--', label=f'Mean   {np.mean(dices):.3f}')
    ax.axvline(np.median(dices), color='green', linestyle='--', label=f'Median {np.median(dices):.3f}')
    ax.set(xlabel='Mean foreground Dice (per sample)',
           ylabel='# test samples',
           title=f'{cfg["dataset"]} — test-set Dice distribution (n={n})')
    ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{cfg["dataset"].lower()}_dice_hist.png', dpi=150)
    plt.show()

dices = [r['dice'] for r in records]
print(f'\n── {cfg["dataset"]} test summary ──')
print(f'  n samples : {n}')
print(f'  Dice mean : {np.mean(dices):.4f}   std {np.std(dices):.4f}')
print(f'  Dice range: [{min(dices):.4f}, {max(dices):.4f}]')

## 10 · Summary JSON

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)